<div dir="rtl" align="right">

# قاعده زنجیره‌ای و گراف محاسبه

درسی از مدرسه پایتون و هوش مصنوعی، دکتر بهروز طاهری

## مسئله
شبکه عصبی یک تابع تو در تو است: خروجی هر لایه ورودی لایه بعد می‌شود. برای آموزش باید بدانیم خطا نسبت به وزنی که چند لایه عقب‌تر است چقدر حساس است. **قاعده زنجیره‌ای** همین را می‌دهد.

## قاعده
اگر $y=f(u)$ و $u=g(x)$ باشد:

$$\frac{dy}{dx}=\frac{dy}{du}\cdot\frac{du}{dx}$$

یعنی: حساسیت‌ها در طول مسیر **در هم ضرب می‌شوند**. مثل چرخ‌دنده: اگر چرخ A هر دور، چرخ B را 3 دور بچرخاند و B هر دور، C را 2 دور، آنگاه A هر دور C را $3\times2=6$ دور می‌چرخاند.

## مثال عددی
$y=(3x+1)^2$ را به دو قدم می‌شکنیم: $u=3x+1$ و $y=u^2$.

| قدم | فرمول | در $x=1$ |
|---|---|---|
| رفت: u | $3x+1$ | 4 |
| رفت: y | $u^2$ | 16 |
| $dy/du$ | $2u$ | 8 |
| $du/dx$ | 3 | 3 |
| $dy/dx$ | $2u\times3$ | 24 |

بررسی: $y=9x^2+6x+1\Rightarrow y'=18x+6=24$ ✓.

## گراف محاسبه: یک نورون
یک نورون با خطای مربعی را به قدم‌های ساده می‌شکنیم:

$$z=wx+b\qquad a=\sigma(z)\qquad L=(a-y)^2$$

> آزمایشگاه تعاملی این بخش را در سایت باز کن: https://behrooztaheri.github.io/learn-python-ai/#/l/math-chain

با $x=2,\ w=0.5,\ b=-0.5,\ y=1$:

| گذر رفت (forward) | مقدار | گذر برگشت (backward) | مقدار |
|---|---|---|---|
| $z=wx+b$ | 0.5 | $\partial L/\partial a=2(a-y)$ | −0.755 |
| $a=\sigma(z)$ | 0.622 | $\partial a/\partial z=a(1-a)$ | 0.235 |
| $L=(a-y)^2$ | 0.143 | $\partial L/\partial z=-0.755\times0.235$ | −0.177 |
| | | $\partial L/\partial w=\partial L/\partial z\times x$ | −0.355 |
| | | $\partial L/\partial b=\partial L/\partial z\times1$ | −0.177 |

> توجه کن که $\partial L/\partial z$ یک بار حساب شد و برای هر دو مشتق w و b **دوباره استفاده** شد. پس‌انتشار خطا (backpropagation) همین است: قاعده زنجیره‌ای + ذخیره نتیجه‌های میانی. به همین دلیل محاسبه گرادیان میلیون‌ها وزن تقریباً به اندازه یک گذر رفت هزینه دارد.

## چند مسیر: جمع کن
اگر یک متغیر از **چند مسیر** روی خروجی اثر بگذارد، سهم همه مسیرها جمع می‌شود:

$$\frac{\partial L}{\partial x}=\sum_{\text{paths}}\frac{\partial L}{\partial u_k}\frac{\partial u_k}{\partial x}$$

مثال: $L=u+v$ با $u=x^2$ و $v=3x$. آنگاه $dL/dx=1\times2x+1\times3=2x+3$.

## مشتق‌های محلی گره‌های رایج
| گره | رفت | برگشت: گرادیان ورودی‌ها |
|---|---|---|
| جمع $c=a+b$ | جمع | گرادیان را بدون تغییر به هر دو می‌دهد |
| ضرب $c=ab$ | ضرب | به a می‌دهد $g\cdot b$ و به b می‌دهد $g\cdot a$ (جابه‌جا) |
| max | بزرگ‌تر | فقط به ورودی بزرگ‌تر می‌رسد |
| ReLU | $\max(0,z)$ | اگر $z>0$ عبور می‌کند، وگرنه صفر |
| سیگموید | $\sigma(z)$ | ضرب در $a(1-a)$ |

## مشتق‌گیری خودکار (autograd)
کتابخانه‌هایی مثل PyTorch و TensorFlow همین کار را خودکار انجام می‌دهند: هنگام محاسبه، گراف را می‌سازند و سپس از آخر به اول قاعده زنجیره‌ای را اعمال می‌کنند. کد زیر یک نسخه 30 خطی از این ایده است:



</div>

In [ ]:
import math

class Value:
    """A number that remembers how it was computed, so it can backpropagate."""
    def __init__(self, data, parents=(), name=""):
        self.data, self.grad, self.parents, self.name = data, 0.0, parents, name
        self._backward = lambda: None
    def __add__(self, other):
        out = Value(self.data + other.data, (self, other))
        def back():
            self.grad += out.grad; other.grad += out.grad
        out._backward = back; return out
    def __mul__(self, other):
        out = Value(self.data * other.data, (self, other))
        def back():
            self.grad += other.data * out.grad; other.grad += self.data * out.grad
        out._backward = back; return out
    def sigmoid(self):
        s = 1 / (1 + math.exp(-self.data)); out = Value(s, (self,))
        def back():
            self.grad += s * (1 - s) * out.grad
        out._backward = back; return out
    def sq_error(self, y):
        out = Value((self.data - y) ** 2, (self,))
        def back():
            self.grad += 2 * (self.data - y) * out.grad
        out._backward = back; return out
    def backward(self):
        order, seen = [], set()
        def visit(v):
            if v not in seen:
                seen.add(v)
                for p in v.parents: visit(p)
                order.append(v)
        visit(self); self.grad = 1.0
        for v in reversed(order): v._backward()

x, w, b = Value(2.0), Value(0.5), Value(-0.5)
z = w * x + b
a = z.sigmoid()
L = a.sq_error(1.0)
L.backward()
print(f"z = {z.data:.3f}  a = {a.data:.3f}  L = {L.data:.4f}")
print(f"dL/dz = {z.grad:.4f}  dL/dw = {w.grad:.4f}  dL/db = {b.grad:.4f}  dL/dx = {x.grad:.4f}")

# Check dL/dw with a numerical derivative
def loss(wv):
    a = 1 / (1 + math.exp(-(wv * 2.0 - 0.5))); return (a - 1.0) ** 2
h = 1e-6
print("numeric dL/dw =", round((loss(0.5 + h) - loss(0.5 - h)) / (2 * h), 4))

<div dir="rtl" align="right">

### تمرین

برای $y=\ln(1+e^{2x})$ با قاعده زنجیره‌ای مشتق بگیر و مقدار آن را در $x=0$ حساب کن.

</div>

In [ ]:
# Your answer here

<div dir="rtl" align="right">

### تمرین

در آزمایشگاه چند بار دکمه «یک گام گرادیان کاهشی» را بزن. L چه می‌شود و چرا؟

</div>

In [ ]:
# Your answer here